# Stage 07 — Generative Spectrum-to-SMILES Transformer
Generates novel structures alongside Stage 03/06 retrieval. Attach Stage 01 clean
data, Stage 02 splits/evalkit, and a **train-only** Stage 05/05b checkpoint.
Select two T4 GPUs. Trains for 10 epochs under a 10h total wall-clock budget (reserving
1.2h for batched beam search and submission validation); epoch one freezes the transferred encoder.
Predicted 180-dim substructure features (element counts, MACCS keys, heavy atom & ring counts)
condition both training and inference. Generated beams undergo Self-Spec2FP discriminative
reranking (Morgan & MACCS Tanimoto, elemental L1, ppm error) and tautomer-canonical deduplication.
Raw spectra are streamed to compact disk-backed records; K=4 randomized SMILES variants.

In [1]:
!pip install -q rdkit

In [2]:
import gc, hashlib, json, os, random, shutil, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
import torch
from torch import nn
from rdkit import Chem, RDLogger
WORK = Path('/kaggle/working')
OUT = WORK / 'artifacts/spec2smiles'
CACHE = WORK / 'spec2smiles_cache'
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK))
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
RDLogger.DisableLog('rdApp.*')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Training requires a Kaggle GPU accelerator'
print('GPUs:', torch.cuda.device_count())
START = time.time()

def check_ram():
    rss = psutil.Process().memory_info().rss / 1e9
    if rss >= 24.0:
        raise MemoryError(f'Host RSS {rss:.2f} GB exceeds 24 GB budget')
    return rss

def find_file(filename, root=Path('/kaggle/input'), hint=''):
    hits = list(Path(root).rglob(filename))
    if not hits:
        raise FileNotFoundError(f'{filename} not found under {root}')
    hits.sort(key=lambda p: (bool(hint and hint in str(p)), p.stat().st_mtime, str(p)),
              reverse=True)
    return hits[0]

GPUs: 2


In [3]:
module_path = WORK / 'spec2fp.py'
module_path.write_text('', encoding='utf-8')

0

In [4]:
fragment = (
'"""spec2fp.py - Spectrum-to-Fingerprint model, tensorization, and inference.\n'
'\n'
'Defines the Spec2FP Transformer, peak tensorization, and predict() API.\n'
'Import ONLY from /kaggle/working/.\n'
'"""\n'
'import json, math, sys, warnings, gc\n'
'from pathlib import Path\n'
'from typing import Dict, List, Optional, Tuple, Union, Any\n'
'import numpy as np\n'
'import pandas as pd\n'
'import torch\n'
'import torch.nn as nn\n'
'import torch.nn.functional as F\n'
'\n'
'ADDUCT_VOCAB = [\n'
'    "<pad>", "[M+H]+", "[M-H]-", "[M+Na]+", "[M+NH4]+", "[M+K]+",\n'
'    "[M+H-H2O]+", "[M+FA-H]-", "[M+Cl]-", "[M]+", "[M-H2O+H]+",\n'
'    "[M-H2O-H]-", "[M+2H]2+", "[M-2H]2-", "[M+HAc-H]-",\n'
'    "[M+2Na-H]+", "[M+CH2O2-H]-", "<unk>",\n'
']\n'
'ADDUCT2IDX = {a: i for i, a in enumerate(ADDUCT_VOCAB)}\n'
'INST_VOCAB = [\n'
'    "<pad>", "timsTOF", "qtof", "orbitrap", "FTICR", "QqQ",\n'
'    "ion_trap", "None/Unknown", "<unk>",\n'
']\n'
'INST2IDX = {a: i for i, a in enumerate(INST_VOCAB)}\n'
'POLARITY_VOCAB = ["<pad>", "positive", "negative", "<unk>"]\n'
'POLARITY2IDX = {a: i for i, a in enumerate(POLARITY_VOCAB)}\n'
'ELEMENTS_ORDER = ["C", "H", "N", "O", "S", "P", "F", "Cl", "Br", "I"]\n'
'N_ELEMENTS = len(ELEMENTS_ORDER)\n'
'\n'
'\n'
'def parse_peak_array(val: Any) -> np.ndarray:\n'
'    """Robustly parse peak m/z or intensity into a float64 numpy array."""\n'
'    if isinstance(val, np.ndarray):\n'
'        return val.astype(np.float64)\n'
'    if isinstance(val, (list, tuple)):\n'
'        return np.asarray(val, dtype=np.float64)\n'
'    if isinstance(val, str):\n'
'        try:\n'
'            return np.asarray(json.loads(val), dtype=np.float64)\n'
'        except Exception:\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [5]:
fragment = (
'            pass\n'
'    try:\n'
'        return np.asarray(val, dtype=np.float64)\n'
'    except Exception:\n'
'        return np.zeros(0, dtype=np.float64)\n'
'\n'
'\n'
'def sinusoidal_embed(values: np.ndarray, dim: int = 64) -> np.ndarray:\n'
'    """Multi-scale sinusoidal encoding resolving 0.005 Da mass defects up to 2000 Da."""\n'
'    freqs = np.logspace(\n'
'        math.log10(2 * math.pi / 2000.0),\n'
'        math.log10(2 * math.pi / 0.005),\n'
'        dim // 2, dtype=np.float32)\n'
'    angles = np.outer(values.astype(np.float32), freqs)\n'
'    return np.concatenate([np.sin(angles), np.cos(angles)], axis=-1)\n'
'\n'
'\n'
'def tensorize_spectrum(\n'
'    mz: Any, inten: Any, precursor_mz: float,\n'
'    adduct: Any, ce: float, instrument: Any, polarity: Any,\n'
'    top_k: int = 192, mz_dim: int = 64,\n'
') -> Dict[str, np.ndarray]:\n'
'    """Convert a single spectrum into model-ready numpy arrays."""\n'
'    mz = parse_peak_array(mz)\n'
'    inten = parse_peak_array(inten)\n'
'    mask_valid = mz <= precursor_mz + 5.0\n'
'    mz, inten = mz[mask_valid], inten[mask_valid]\n'
'    if len(mz) == 0:\n'
'        mz = np.array([precursor_mz], dtype=np.float64)\n'
'        inten = np.array([1.0], dtype=np.float64)\n'
'    max_int = max(inten.max(), 1e-12)\n'
'    inten = np.sqrt(inten / max_int)\n'
'    if len(mz) > top_k:\n'
'        idx = np.argsort(inten)[-top_k:]\n'
'        idx = idx[np.argsort(mz[idx])]\n'
'        mz, inten = mz[idx], inten[idx]\n'
'    n = len(mz)\n'
'    mz_emb = sinusoidal_embed(mz, mz_dim)\n'
'    nl = precursor_mz - mz\n'
'    nl_emb = sinusoidal_embed(nl, mz_dim)\n'
'    log_int = np.log1p(inten).astype(np.float32).reshape(-1, 1)\n'
'    peak_feat = np.concatenate([mz_emb, log_int, nl_emb], axis=1).astype(np.float32)\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [6]:
fragment = (
'    adduct_idx = int(adduct) if isinstance(adduct, (int, np.integer)) else ADDUCT2IDX.get(str(adduct).strip(), ADDUCT2IDX["<unk>"])\n'
'    inst_idx = int(instrument) if isinstance(instrument, (int, np.integer)) else INST2IDX.get(str(instrument).strip(), INST2IDX["<unk>"])\n'
'    pol_idx = int(polarity) if isinstance(polarity, (int, np.integer)) else POLARITY2IDX.get(str(polarity).strip().lower(), POLARITY2IDX["<unk>"])\n'
'    ce_val = float(ce) if not (ce is None or (isinstance(ce, float) and math.isnan(ce))) else 0.0\n'
'    return {\n'
'        "peak_feat": peak_feat, "n_peaks": n, "mz": mz,\n'
'        "adduct_idx": adduct_idx, "inst_idx": inst_idx, "pol_idx": pol_idx,\n'
'        "ce_val": np.float32(ce_val), "prec_mz": np.float32(precursor_mz),\n'
'    }\n'
'\n'
'\n'
'def collate_spectra(batch: list, top_k: int = 192, feat_dim: int = 129):\n'
'    """Collate list of tensorize_spectrum outputs into batched tensors."""\n'
'    B = len(batch)\n'
'    max_len = top_k + 3\n'
'    peaks = torch.zeros(B, max_len, feat_dim, dtype=torch.float32)\n'
'    mask = torch.ones(B, max_len, dtype=torch.bool)\n'
'    adducts = torch.zeros(B, dtype=torch.long)\n'
'    instruments = torch.zeros(B, dtype=torch.long)\n'
'    polarities = torch.zeros(B, dtype=torch.long)\n'
'    ces = torch.zeros(B, dtype=torch.float32)\n'
'    prec_mzs = torch.zeros(B, dtype=torch.float32)\n'
'    for i, item in enumerate(batch):\n'
'        n = item["n_peaks"]\n'
'        peaks[i, :n] = torch.from_numpy(item["peak_feat"][:n])\n'
'        mask[i, :n] = False\n'
'        adducts[i] = item["adduct_idx"]\n'
'        instruments[i] = item["inst_idx"]\n'
'        polarities[i] = item["pol_idx"]\n'
'        ces[i] = float(item["ce_val"])\n'
'        prec_mzs[i] = float(item["prec_mz"])\n'
'    return {\n'
'        "peaks": peaks, "pad_mask": mask,\n'
'        "adduct": adducts, "instrument": instruments, "polarity": polarities,\n'
'        "ce": ces, "prec_mz": prec_mzs,\n'
'    }\n'
'\n'
'\n'
'class Spec2FPModel(nn.Module):\n'
'    """Transformer encoder mapping MS/MS spectrum to molecular fingerprint."""\n'
'\n'
'    def __init__(self, cfg: dict):\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [7]:
fragment = (
'        super().__init__()\n'
'        self.cfg = cfg\n'
'        d, n_layers, n_heads = cfg.get("d_model", 320), cfg.get("n_layers", 5), cfg.get("n_heads", 8)\n'
'        dropout, feat_dim = cfg.get("dropout", 0.1), cfg.get("feat_dim", 129)\n'
'        fp_bits, maccs_bits = cfg.get("fp_bits", 4096), cfg.get("maccs_bits", 167)\n'
'        n_elements = cfg.get("n_elements", N_ELEMENTS)\n'
'        self.peak_proj = nn.Sequential(nn.Linear(feat_dim, d), nn.GELU(), nn.LayerNorm(d), nn.Dropout(dropout))\n'
'        self.cls_token = nn.Parameter(torch.randn(1, 1, d) * 0.02)\n'
'        self.adduct_emb = nn.Embedding(len(ADDUCT_VOCAB), d, padding_idx=0)\n'
'        self.inst_emb = nn.Embedding(len(INST_VOCAB), d, padding_idx=0)\n'
'        self.pol_emb = nn.Embedding(len(POLARITY_VOCAB), d, padding_idx=0)\n'
'        self.ce_proj = nn.Sequential(nn.Linear(1, d), nn.GELU())\n'
'        self.prec_proj = nn.Sequential(nn.Linear(1, d), nn.GELU())\n'
'        enc_layer = nn.TransformerEncoderLayer(\n'
'            d_model=d, nhead=n_heads, dim_feedforward=d * 4,\n'
'            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)\n'
'        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)\n'
'        self.norm = nn.LayerNorm(d)\n'
'        self.fp_head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, fp_bits))\n'
'        self.maccs_head = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(), nn.Linear(d // 2, maccs_bits))\n'
'        self.heavy_atom_head = nn.Linear(d, 1)\n'
'        self.ring_count_head = nn.Linear(d, 1)\n'
'        self.elem_presence_head = nn.Linear(d, n_elements)\n'
'        self.elem_count_head = nn.Linear(d, n_elements)\n'
'\n'
'    def forward(self, peaks, pad_mask, adduct, instrument, polarity, ce, prec_mz):\n'
'        B = peaks.size(0)\n'
'        x = self.peak_proj(peaks)\n'
'        cls = self.cls_token.expand(B, -1, -1)\n'
'        meta1 = (\n'
'            self.adduct_emb(adduct)\n'
'            + self.ce_proj(ce.unsqueeze(-1) / 100.0)\n'
'            + self.prec_proj(prec_mz.unsqueeze(-1) / 1000.0)\n'
'        )\n'
'        meta2 = self.inst_emb(instrument) + self.pol_emb(polarity)\n'
'        meta = torch.stack([meta1, meta2], dim=1)\n'
'        x = torch.cat([cls, meta, x], dim=1)\n'
'        cls_meta_mask = torch.zeros(B, 3, dtype=torch.bool, device=pad_mask.device)\n'
'        full_mask = torch.cat([cls_meta_mask, pad_mask], dim=1)\n'
'        x = self.encoder(x, src_key_padding_mask=full_mask)\n'
'        cls_out = x[:, 0]\n'
'        valid_tok = (~full_mask).float().unsqueeze(-1)\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [8]:
fragment = (
'        mean_out = (x * valid_tok).sum(dim=1) / valid_tok.sum(dim=1).clamp(min=1.0)\n'
'        x = self.norm(cls_out + mean_out)\n'
'        return {\n'
'            "fp_logits": self.fp_head(x), "maccs_logits": self.maccs_head(x),\n'
'            "heavy_atoms": self.heavy_atom_head(x).squeeze(-1),\n'
'            "ring_count": self.ring_count_head(x).squeeze(-1),\n'
'            "elem_presence": self.elem_presence_head(x),\n'
'            "elem_count": self.elem_count_head(x),\n'
'        }\n'
'\n'
'\n'
'def load_model(config_path: str, weights_path: str, device: str = "cpu"):\n'
'    """Load Spec2FP model from config + weights."""\n'
'    with open(config_path) as f:\n'
'        cfg = json.load(f)\n'
'    model = Spec2FPModel(cfg["model"])\n'
'    sd = torch.load(weights_path, map_location=device, weights_only=False)\n'
'    if "ema_state_dict" in sd:\n'
'        model.load_state_dict({k.replace("module.", ""): v for k, v in sd["ema_state_dict"].items()})\n'
'    elif "model_state_dict" in sd:\n'
'        model.load_state_dict({k.replace("module.", ""): v for k, v in sd["model_state_dict"].items()})\n'
'    else:\n'
'        model.load_state_dict({k.replace("module.", ""): v for k, v in sd.items()})\n'
'    pw_path = Path(weights_path).parent / "spec2fp_pos_weights.npy"\n'
'    if pw_path.is_file():\n'
'        try:\n'
'            model.pos_weights = np.load(pw_path)\n'
'        except Exception:\n'
'            pass\n'
'    model.eval()\n'
'    return model.to(device), cfg\n'
'\n'
'\n'
'@torch.no_grad()\n'
'def predict_full(model, clean_df, device="cuda", batch_size=128, top_k=192, mz_dim=64, pos_weights=None):\n'
'    """Predict Morgan-4096, MACCS-167, and element counts with streaming molecule-level pooling."""\n'
'    raw_model = model.module if hasattr(model, "module") else model\n'
'    raw_model.eval()\n'
'    pw = pos_weights if pos_weights is not None else getattr(raw_model, "pos_weights", None)\n'
'    pw_shift = 0.5 * torch.log(torch.as_tensor(pw, device=device, dtype=torch.float32).clamp(min=1.0)) if pw is not None else None\n'
'    feat_dim, n_rows = 2 * mz_dim + 1, len(clean_df)\n'
'    if n_rows == 0:\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [9]:
fragment = (
'        return (np.zeros((0, 4096), dtype=np.float16), np.zeros((0, 167), dtype=np.float16),\n'
'                np.zeros((0, N_ELEMENTS), dtype=np.float32), [])\n'
'\n'
'    mol_ids_arr = clean_df["molecule_id"].to_numpy().astype(str)\n'
'    unique_mols, inv_indices = np.unique(mol_ids_arr, return_inverse=True)\n'
'    n_unique = len(unique_mols)\n'
'    fp_sum = np.zeros((n_unique, 4096), dtype=np.float32)\n'
'    maccs_sum = np.zeros((n_unique, 167), dtype=np.float32)\n'
'    elem_sum = np.zeros((n_unique, N_ELEMENTS), dtype=np.float32)\n'
'    counts = np.zeros(n_unique, dtype=np.float32)\n'
'\n'
'    mz_col = clean_df["mz"].to_numpy()\n'
'    int_col = clean_df["inten"].to_numpy()\n'
'    pmz_col = clean_df["precursor_mz"].to_numpy(dtype=np.float32)\n'
'    add_col = clean_df["adduct"].to_numpy() if "adduct" in clean_df.columns else np.array(["[M+H]+"] * n_rows)\n'
'    ce_col = pd.to_numeric(clean_df["ce"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float32) if "ce" in clean_df.columns else np.zeros(n_rows, dtype=np.float32)\n'
'    inst_col = clean_df["instrument"].to_numpy() if "instrument" in clean_df.columns else np.array(["unknown"] * n_rows)\n'
'    pol_col = clean_df["polarity"].to_numpy() if "polarity" in clean_df.columns else np.array(["positive"] * n_rows)\n'
'    dev_type, is_cuda = ("cuda" if "cuda" in str(device) else "cpu"), ("cuda" in str(device))\n'
'\n'
'    for s_idx in range(0, n_rows, batch_size):\n'
'        e_idx = min(s_idx + batch_size, n_rows)\n'
'        items = [\n'
'            tensorize_spectrum(\n'
'                mz_col[r], int_col[r], float(pmz_col[r]),\n'
'                str(add_col[r]), float(ce_col[r]),\n'
'                str(inst_col[r]), str(pol_col[r]), top_k=top_k, mz_dim=mz_dim)\n'
'            for r in range(s_idx, e_idx)\n'
'        ]\n'
'        bd = collate_spectra(items, top_k=top_k, feat_dim=feat_dim)\n'
'        bd = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in bd.items()}\n'
'        with torch.amp.autocast(device_type=dev_type, dtype=torch.float16, enabled=is_cuda):\n'
'            out = raw_model(**bd)\n'
'        fp_logits = out["fp_logits"] - pw_shift if pw_shift is not None else out["fp_logits"]\n'
'        batch_fp = torch.sigmoid(fp_logits).float().cpu().numpy()\n'
'        batch_maccs = torch.sigmoid(out["maccs_logits"]).float().cpu().numpy()\n'
'        batch_elem = out["elem_count"].float().cpu().numpy()\n'
'        batch_inv = inv_indices[s_idx:e_idx]\n'
'        np.add.at(fp_sum, batch_inv, batch_fp)\n'
'        np.add.at(maccs_sum, batch_inv, batch_maccs)\n'
'        np.add.at(elem_sum, batch_inv, batch_elem)\n'
'        np.add.at(counts, batch_inv, 1.0)\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [10]:
fragment = (
'\n'
'    counts_col = np.maximum(counts, 1.0)[:, None]\n'
'    return ((fp_sum / counts_col).astype(np.float16),\n'
'            (maccs_sum / counts_col).astype(np.float16),\n'
'            (elem_sum / counts_col).astype(np.float32), unique_mols.tolist())\n'
'\n'
'\n'
'@torch.no_grad()\n'
'def predict(model, clean_df, device="cuda", batch_size=128, top_k=192, mz_dim=64, pos_weights=None, return_maccs=False):\n'
'    """Predict fingerprints with optional MACCS return for backward-compatibility."""\n'
'    fp_pred, maccs_pred, elem_pred, mol_ids = predict_full(\n'
'        model, clean_df, device=device, batch_size=batch_size, top_k=top_k, mz_dim=mz_dim, pos_weights=pos_weights\n'
'    )\n'
'    if return_maccs:\n'
'        return fp_pred, maccs_pred, elem_pred, mol_ids\n'
'    return fp_pred, elem_pred, mol_ids\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [11]:
module_path = WORK / 'spec2smiles.py'
module_path.write_text('', encoding='utf-8')

0

In [12]:
fragment = (
'"""Stage 07: transferable spectral encoder, 180-dim substructure conditioning, and Self-Spec2FP reranked SMILES decoder."""\n'
'import copy\n'
'import json\n'
'import math\n'
'import re\n'
'from itertools import islice\n'
'from pathlib import Path\n'
'\n'
'import numpy as np\n'
'import pandas as pd\n'
'import torch\n'
'from torch import nn\n'
'from rdkit import Chem\n'
'from rdkit.Chem import rdMolDescriptors\n'
'import evalkit\n'
'from spec2fp import (Spec2FPModel, ELEMENTS_ORDER, tensorize_spectrum,\n'
'                     collate_spectra, parse_peak_array)\n'
'\n'
"ENCODER_NAMES = ('peak_proj', 'cls_token', 'adduct_emb', 'inst_emb',\n"
"                 'pol_emb', 'ce_proj', 'prec_proj', 'encoder', 'norm')\n"
"AUX_HEAD_NAMES = ('fp_head', 'maccs_head', 'heavy_atom_head',\n"
"                  'ring_count_head', 'elem_presence_head', 'elem_count_head')\n"
"COLUMNS = ['molecule_id', 'smiles', 'ik14', 'fast_key', 'score',\n"
"           'log_prob', 'nb_ll', 'cos_sim', 'tan_sim', 'maccs_cos', 'maccs_tan',\n"
"           'elem_dist', 'ppm_err', 'rank', 'source']\n"
'\n'
"SMILES_PATTERN = r'(\\[[^\\]]+]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\\(|\\)|\\.|=|#|-|\\+|\\\\|\\/|:|~|@|\\?|>|\\*|\\$|\\%[0-9]{2}|[0-9])'\n"
'\n'
'\n'
'class SmilesTokenizer:\n'
'    def __init__(self, vocab=None, max_length=128):\n'
"        self.vocab = vocab or dict(zip(('<pad>', '<bos>', '<eos>', '<unk>'), range(4)))\n"
'        self.max_length = max_length\n'
'        self.pattern = re.compile(SMILES_PATTERN)\n'
'        self.inverse = {v: k for k, v in self.vocab.items()}\n'
'\n'
'    def tokens(self, smiles):\n'
'        tokens = self.pattern.findall(smiles)\n'
"        if ''.join(tokens) != smiles:\n"
"            raise ValueError(f'Unrecognized SMILES syntax: {smiles}')\n"
'        return tokens\n'
'\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [13]:
fragment = (
'    def fit(self, smiles):\n'
'        found = set()\n'
'        for smi in smiles:\n'
'            found.update(self.tokens(smi))\n'
"        found.update(list('0123456789') + [f'%{i:02d}' for i in range(10, 30)])\n"
'        self.vocab.update({s: i + 4 for i, s in enumerate(sorted(found))})\n'
'        self.inverse = {v: k for k, v in self.vocab.items()}\n'
'        return self\n'
'\n'
'    def encode(self, smiles):\n'
'        tokens = self.tokens(smiles)\n'
'        if len(tokens) + 2 > self.max_length:\n'
"            raise ValueError('SMILES exceeds sequence budget; never truncate molecules')\n"
'        ids = [1] + [self.vocab.get(t, 3) for t in tokens] + [2]\n'
'        return ids + [0] * (self.max_length - len(ids))\n'
'\n'
'    def decode(self, ids):\n'
'        out = []\n'
'        for idx in ids:\n'
'            if int(idx) == 2:\n'
'                break\n'
'            if int(idx) not in (0, 1):\n'
'                out.append(self.inverse[int(idx)])\n'
"        return ''.join(out)\n"
'\n'
'    def save(self, path):\n'
'        Path(path).write_text(json.dumps(dict(vocab=self.vocab,\n'
'                                             max_length=self.max_length), indent=2))\n'
'\n'
'    @classmethod\n'
'    def load(cls, path):\n'
'        return cls(**json.loads(Path(path).read_text()))\n'
'\n'
'\n'
'def mol_elem_counts(mol):\n'
'    counts = np.zeros(len(ELEMENTS_ORDER), dtype=np.float32)\n'
'    mol_h = Chem.AddHs(mol)\n'
'    for atom in mol_h.GetAtoms():\n'
'        sym = atom.GetSymbol()\n'
'        if sym in ELEMENTS_ORDER:\n'
'            counts[ELEMENTS_ORDER.index(sym)] += 1.0\n'
'    return counts\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [14]:
fragment = (
'\n'
'\n'
'class Spec2SmilesModel(nn.Module):\n'
'    def __init__(self, cfg):\n'
'        super().__init__()\n'
'        self.cfg = cfg\n'
"        base = Spec2FPModel(cfg['encoder'])\n"
'        for name in ENCODER_NAMES:\n'
'            setattr(self, name, getattr(base, name))\n'
'        # Retain all pretrained Stage 05b auxiliary heads (frozen) for conditioning & reranking\n'
'        for name in AUX_HEAD_NAMES:\n'
'            head = getattr(base, name)\n'
'            head.requires_grad_(False)\n'
'            setattr(self, name, head)\n'
'\n'
"        enc_dim = cfg['encoder']['d_model']\n"
'        self.memory_proj = nn.Linear(enc_dim, 320) if enc_dim != 320 else nn.Identity()\n'
'        # 180-dim conditioning: neutral_mass(1) + elem_counts(10) + maccs_probs(167) + heavy(1) + rings(1)\n'
'        self.condition = nn.Sequential(\n'
'            nn.Linear(180, 320), nn.GELU(), nn.LayerNorm(320),\n'
'            nn.Linear(320, 320), nn.GELU(), nn.LayerNorm(320)\n'
'        )\n'
'        self.pooled_proj = nn.Sequential(nn.Linear(enc_dim, 320), nn.LayerNorm(320))\n'
"        self.token_emb = nn.Embedding(cfg['vocab_size'], 320, padding_idx=0)\n"
'        nn.init.normal_(self.token_emb.weight, std=0.02)\n'
'        with torch.no_grad():\n'
'            self.token_emb.weight[0].zero_()\n'
'        layer = nn.TransformerDecoderLayer(320, 8, 1280, 0.1,\n'
'                                           batch_first=True, norm_first=True)\n'
"        self.decoder = nn.TransformerDecoder(layer, cfg.get('decoder_layers', 4), norm=nn.LayerNorm(320))\n"
"        self.output = nn.Linear(320, cfg['vocab_size'], bias=False)\n"
'        self.output.weight = self.token_emb.weight\n'
'        pos = torch.arange(128).float().unsqueeze(1)\n'
'        angles = pos * torch.exp(torch.arange(0, 320, 2).float() * (-math.log(10000)/320))\n'
'        pe = torch.zeros(128, 320)\n'
'        pe[:, 0::2], pe[:, 1::2] = angles.sin(), angles.cos()\n'
"        self.register_buffer('positions', pe)\n"
"        self.register_buffer('pw_shift', torch.zeros(4096))\n"
'\n'
'    def encoder_parameters(self):\n'
"        return [p for n, p in self.named_parameters() if n.split('.')[0] in ENCODER_NAMES]\n"
'\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [15]:
fragment = (
'    def warm_start(self, path):\n'
"        state = torch.load(path, map_location='cpu', weights_only=False)\n"
"        state = state.get('ema_state_dict', state.get('model_state_dict', state))\n"
"        state = {k.removeprefix('module.'): v for k, v in state.items()}\n"
'        wanted = {k: v for k, v in self.state_dict().items()\n'
"                  if k.split('.')[0] in ENCODER_NAMES + AUX_HEAD_NAMES}\n"
'        missing = set(wanted) - set(state)\n'
'        if missing:\n'
"            raise ValueError(f'Incomplete pretrained encoder: {sorted(missing)}')\n"
'        self.load_state_dict({k: state[k] for k in wanted}, strict=False)\n'
"        pw_path = Path(path).parent / 'spec2fp_pos_weights.npy'\n"
'        if pw_path.is_file():\n'
'            pw = torch.from_numpy(np.load(pw_path)).float().clamp(min=1.0)\n'
'            self.pw_shift.copy_(0.5 * torch.log(pw))\n'
'\n'
'    def encode(self, peaks, pad_mask, adduct, instrument, polarity, ce, prec_mz, return_fp=False):\n'
'        b = len(peaks)\n'
'        meta1 = self.adduct_emb(adduct) + self.ce_proj(ce[:, None]/100)\n'
'        meta1 = meta1 + self.prec_proj(prec_mz[:, None]/1000)\n'
'        meta2 = self.inst_emb(instrument) + self.pol_emb(polarity)\n'
'        x = torch.cat((self.cls_token.expand(b, -1, -1),\n'
'                       torch.stack((meta1, meta2), 1), self.peak_proj(peaks)), 1)\n'
'        mask = torch.cat((pad_mask.new_zeros((b, 3)), pad_mask), 1)\n'
'        x = self.encoder(x, src_key_padding_mask=mask)\n'
'        valid = (~mask).unsqueeze(-1)\n'
'        pooled = self.norm(x[:, 0] + (x*valid).sum(1)/valid.sum(1).clamp_min(1))\n'
'        with torch.no_grad():\n'
'            counts = self.elem_count_head(pooled).detach().clamp(0, 200)\n'
'            maccs_probs = torch.sigmoid(self.maccs_head(pooled)).detach()\n'
'            heavy = self.heavy_atom_head(pooled).detach().clamp(1, 150) / 100.0\n'
'            rings = self.ring_count_head(pooled).detach().clamp(0, 20) / 10.0\n'
'            cond_vec = torch.cat((counts / 50.0, maccs_probs, heavy, rings), dim=-1)  # 179 dims\n'
'            if return_fp:\n'
'                fp_probs = torch.sigmoid(self.fp_head(pooled) - self.pw_shift).detach()\n'
'                return self.memory_proj(self.norm(x)), mask, pooled, cond_vec, counts, fp_probs, maccs_probs\n'
'        return self.memory_proj(self.norm(x)), mask, pooled, cond_vec\n'
'\n'
'    def conditioned_memory(self, memory, mask, neutral_mass, pooled, cond_vec):\n'
'        full_cond = torch.cat((neutral_mass[:, None] / 1000.0, cond_vec), dim=-1)  # 180 dims\n'
'        cond_tok = self.condition(full_cond) + self.pooled_proj(pooled)\n'
'        return torch.cat((cond_tok[:, None], memory), 1), torch.cat((mask[:, :1]*False, mask), 1)\n'
'\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [16]:
fragment = (
'    def decode(self, tokens, memory, mask):\n'
'        x = self.token_emb(tokens)*math.sqrt(320) + self.positions[:tokens.shape[1]]\n'
'        causal = torch.ones(tokens.shape[1], tokens.shape[1], device=x.device,\n'
'                            dtype=torch.bool).triu(1)\n'
'        x = self.decoder(x, memory, tgt_mask=causal, tgt_key_padding_mask=tokens.eq(0),\n'
'                         memory_key_padding_mask=mask)\n'
'        return self.output(x)\n'
'\n'
'    def forward(self, tokens, neutral_mass, **spectra):\n'
'        memory, mask, pooled, cond_vec = self.encode(**spectra)\n'
'        memory, mask = self.conditioned_memory(memory, mask, neutral_mass, pooled, cond_vec)\n'
'        return self.decode(tokens, memory, mask)\n'
'\n'
'\n'
'def spectrum_item(row, cfg):\n'
'    return tensorize_spectrum(row.mz, row.inten, float(row.precursor_mz),\n'
'                             row.adduct, float(row.ce), row.instrument, row.polarity,\n'
"                             top_k=cfg['top_k_peaks'], mz_dim=cfg['mz_dim'])\n"
'\n'
'\n'
'def record_dtype(top_k):\n'
"    return np.dtype([('mz', 'f4', (top_k,)), ('inten', 'f4', (top_k,)),\n"
"                     ('n', 'i2'), ('prec', 'f4'), ('mass', 'f4'), ('ce', 'f4'),\n"
"                     ('adduct', 'i2'), ('inst', 'i2'), ('pol', 'i2'), ('target', 'i4')])\n"
'\n'
'\n'
'class SpectrumDataset(torch.utils.data.Dataset):\n'
'    """Raw top peaks on disk; supports K-variant precomputed SMILES augmentation."""\n'
'    def __init__(self, path, cfg, tokenizer, smiles, tokens, augment=False, tokens_aug=None):\n'
'        self.cfg, self.tokenizer = cfg, tokenizer\n'
'        self.smiles, self.tokens, self.augment = smiles, tokens, augment\n'
'        self.tokens_aug = tokens[None, ...] if tokens_aug is None else tokens_aug\n'
'        if self.tokens_aug.ndim == 2:\n'
'            self.tokens_aug = self.tokens_aug[None, ...]\n'
"        self.rows = np.memmap(path, mode='r', dtype=record_dtype(cfg['top_k_peaks']))\n"
'\n'
'    def __len__(self):\n'
'        return len(self.rows)\n'
'\n'
'    def __getitem__(self, idx):\n'
'        r = self.rows[idx]\n'
"        t_idx = int(r['target'])\n"
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [17]:
fragment = (
'        if self.augment and np.random.random() < 0.65:\n'
'            k = np.random.randint(0, self.tokens_aug.shape[0])\n'
'            target = self.tokens_aug[k, t_idx].astype(np.int64)\n'
'        else:\n'
'            target = self.tokens[t_idx].astype(np.int64)\n'
"        item = tensorize_spectrum(r['mz'][:r['n']], r['inten'][:r['n']], r['prec'],\n"
"                                  int(r['adduct']), r['ce'], int(r['inst']), int(r['pol']),\n"
"                                  self.cfg['top_k_peaks'], self.cfg['mz_dim'])\n"
"        return item, float(r['mass']), target\n"
'\n'
'    def collate(self, rows):\n'
'        items, masses, tokens = zip(*rows)\n'
"        batch = collate_spectra(items, self.cfg['top_k_peaks'], 2*self.cfg['mz_dim']+1)\n"
"        batch['neutral_mass'] = torch.tensor(masses, dtype=torch.float32)\n"
'        return batch, torch.from_numpy(np.stack(tokens))\n'
'\n'
'\n'
'def append_records(frame, stream, cfg, target_map):\n'
'    from spec2fp import ADDUCT2IDX, INST2IDX, POLARITY2IDX\n'
"    records = np.zeros(len(frame), dtype=record_dtype(cfg['top_k_peaks']))\n"
'    for i, row in enumerate(frame.itertuples(index=False)):\n'
'        if not (math.isfinite(row.neutral_mass) and row.neutral_mass > 0\n'
'                and math.isfinite(row.precursor_mz) and row.precursor_mz > 0):\n'
"            raise ValueError('Training spectra require finite positive precursor/neutral masses')\n"
'        mz, inten = parse_peak_array(row.mz), parse_peak_array(row.inten)\n'
'        valid = np.isfinite(mz) & np.isfinite(inten) & (inten >= 0)\n'
'        valid &= mz <= float(row.precursor_mz) + 5\n'
'        mz, inten = mz[valid], inten[valid]\n'
"        idx = np.argsort(inten)[-cfg['top_k_peaks']:]\n"
'        idx = idx[np.argsort(mz[idx])]\n'
'        n = len(idx)\n'
"        records[i]['mz'][:n], records[i]['inten'][:n] = mz[idx], inten[idx]\n"
"        records[i]['n'], records[i]['prec'] = n, row.precursor_mz\n"
"        records[i]['mass'], records[i]['ce'] = row.neutral_mass, row.ce\n"
"        records[i]['adduct'] = ADDUCT2IDX.get(str(row.adduct).strip(), ADDUCT2IDX['<unk>'])\n"
"        records[i]['inst'] = INST2IDX.get(str(row.instrument).strip(), INST2IDX['<unk>'])\n"
"        records[i]['pol'] = POLARITY2IDX.get(str(row.polarity).strip().lower(), POLARITY2IDX['<unk>'])\n"
"        records[i]['target'] = target_map[str(row.molecule_id)]\n"
'    records.tofile(stream)\n'
'\n'
'\n'
'class EMA:\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [18]:
fragment = (
'    def __init__(self, model, decay=0.999):\n'
'        self.model = copy.deepcopy(model).eval().requires_grad_(False)\n'
'        self.decay = decay\n'
'\n'
'    @torch.no_grad()\n'
'    def update(self, model):\n'
'        for dst, src in zip(self.model.parameters(), model.parameters()):\n'
'            dst.lerp_(src, 1-self.decay)\n'
'\n'
'\n'
'def run_epoch(model, loader, device, optimizer=None, scaler=None, ema=None, scheduler=None):\n'
'    training = optimizer is not None\n'
'    model.train(training)\n'
'    raw = model.module if isinstance(model, nn.DataParallel) else model\n'
'    for name in AUX_HEAD_NAMES:\n'
'        getattr(raw, name).eval()\n'
'    if training and not raw.cls_token.requires_grad:\n'
'        for name in ENCODER_NAMES:\n'
'            module = getattr(raw, name)\n'
'            if isinstance(module, nn.Module):\n'
'                module.eval()\n'
'    total, count = 0.0, 0\n'
'    for batch, target in loader:\n'
'        batch = {k: v.to(device) for k, v in batch.items()}\n'
'        target = target.to(device)\n'
'        with torch.set_grad_enabled(training):\n'
"            with torch.amp.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):\n"
'                logits = model(tokens=target[:, :-1], **batch)\n'
'                loss = nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]),\n'
'                    target[:, 1:].reshape(-1), ignore_index=0, label_smoothing=0.05)\n'
'            if not torch.isfinite(loss):\n'
"                raise FloatingPointError('Non-finite SMILES loss; check spectra and checkpoint')\n"
'            if training:\n'
'                optimizer.zero_grad(set_to_none=True)\n'
'                scaler.scale(loss).backward()\n'
'                scaler.unscale_(optimizer)\n'
'                nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n'
'                old_scale = scaler.get_scale()\n'
'                scaler.step(optimizer)\n'
'                scaler.update()\n'
'                if scaler.get_scale() >= old_scale:\n'
'                    ema.update(raw)\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [19]:
fragment = (
'                    if scheduler is not None:\n'
'                        scheduler.step()\n'
'        n = int(target[:, 1:].ne(0).sum())\n'
'        total, count = total + float(loss.detach())*n, count+n\n'
'    if not count:\n'
"        raise ValueError('Empty training/validation loader')\n"
'    return total/count\n'
'\n'
'\n'
'@torch.no_grad()\n'
'def beam_search(model, memory, mask, beam_size=16, max_len=124, top_k=25):\n'
'    """Decode B molecules together; return length-penalized completed paths."""\n'
'    if beam_size < 1 or max_len < 2 or max_len > 128 or top_k < 1:\n'
"        raise ValueError('Invalid beam size, sequence budget, or candidate count')\n"
'    b, length, dim = memory.shape\n'
'    k, device = beam_size, memory.device\n'
'    memory = memory[:, None].expand(-1, k, -1, -1).reshape(b*k, length, dim)\n'
'    mask = mask[:, None].expand(-1, k, -1).reshape(b*k, length)\n'
'    seq = torch.ones((b, k, 1), dtype=torch.long, device=device)\n'
'    scores = torch.full((b, k), -torch.inf, device=device)\n'
'    scores[:, 0] = 0\n'
'    archive_size = max(top_k * 10, 256)\n'
'    archive_scores = torch.full((b, archive_size), -torch.inf, device=device)\n'
'    archive = torch.zeros((b, archive_size, max_len), dtype=torch.long, device=device)\n'
'    for step in range(1, max_len):\n'
'        logits = model.decode(seq.reshape(b*k, -1), memory, mask)\n'
'        logp = logits[:, -1].float().log_softmax(-1).reshape(b, k, -1)\n'
'        logp[:, :, [0, 1, 3]] = -torch.inf\n'
'        if step == 1:\n'
'            logp[:, :, 2] = -torch.inf\n'
'        combined = logp + scores[:, :, None]\n'
'        vocab = combined.shape[-1]\n'
'        values, indices = combined.flatten(1).topk(min(2*k, k*vocab), dim=1)\n'
'        parents, tokens = indices // vocab, indices % vocab\n'
'        paths = torch.cat((seq.gather(1, parents[:, :, None].expand(-1, -1, step)),\n'
'                           tokens[:, :, None]), dim=2)\n'
'        eos_scores = values.masked_fill(tokens.ne(2), -torch.inf)\n'
'        norm_eos_scores = eos_scores / ((step + 1) ** 0.7)\n'
'        padded = nn.functional.pad(paths, (0, max_len-step-1))\n'
'        archive_scores, keep = torch.cat((archive_scores, norm_eos_scores), 1).topk(archive_size, dim=1)\n'
'        archive = torch.cat((archive, padded), 1).gather(1, keep[:, :, None].expand(-1, -1, max_len))\n'
'        scores, live = values.masked_fill(tokens.eq(2), -torch.inf).topk(k, dim=1)\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [20]:
fragment = (
'        seq = paths.gather(1, live[:, :, None].expand(-1, -1, step+1))\n'
'        live_upper = scores.max(1).values / (max_len ** 0.7)\n'
'        done = live_upper <= archive_scores[:, -1]\n'
'        scores = scores.masked_fill(done[:, None], -torch.inf)\n'
'        if not torch.isfinite(scores).any():\n'
'            break\n'
'    result = []\n'
'    for paths, values in zip(archive.cpu().tolist(), archive_scores.cpu().tolist()):\n'
'        result.append([(ids[:ids.index(2)+1], score) for ids, score in zip(paths, values)\n'
'                       if math.isfinite(score) and 2 in ids])\n'
'    return result\n'
'\n'
'\n'
'@torch.no_grad()\n'
'def generate_candidates(model, tokenizer, query_df, device, beam_size=16, top_k=25, mol_batch_size=64, max_len=124):\n'
'    model = model.module if isinstance(model, nn.DataParallel) else model\n'
'    model.eval()\n'
'    device = torch.device(device)\n'
'    rows = []\n'
'    if beam_size < 1 or top_k < 1 or mol_batch_size < 1:\n'
"        raise ValueError('beam_size and top_k must be positive')\n"
"    top_k_peaks = model.cfg['top_k_peaks']\n"
"    groups = iter(query_df.groupby('molecule_id', sort=False))\n"
'    while True:\n'
'        chunk = list(islice(groups, mol_batch_size))\n'
'        if not chunk:\n'
'            break\n'
'        memories, masks, masses, pred_fps, pred_maccs, pred_elems = [], [], [], [], [], []\n'
'        for molecule_id, group in chunk:\n'
'            mass = float(group.neutral_mass.median())\n'
'            if not math.isfinite(mass) or mass <= 0:\n'
"                raise ValueError(f'Invalid neutral mass for {molecule_id}')\n"
'            meta_mem_sum, pooled_sum, cond_sum, elem_sum, fp_sum, maccs_sum, n = None, None, None, None, None, None, 0\n'
'            peak_tokens_list, peak_intens_list, peak_mzs_list = [], [], []\n'
"            with torch.amp.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):\n"
'                for start in range(0, len(group), 32):\n'
'                    sub = group.iloc[start:start+32]\n'
'                    items = [spectrum_item(r, model.cfg) for r in sub.itertuples()]\n'
"                    batch = collate_spectra(items, top_k_peaks, 2*model.cfg['mz_dim']+1)\n"
'                    memory, mask, pooled, cond_vec, counts, fp_p, maccs_p = model.encode(\n'
'                        **{k: v.to(device) for k, v in batch.items()}, return_fp=True\n'
'                    )\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [21]:
fragment = (
'                    # Global tokens 0, 1, 2 (CLS, meta1, meta2) are positionally aligned across spectra\n'
'                    meta_mem = memory[:, :3].float().sum(0, keepdim=True)\n'
'                    meta_mem_sum = meta_mem if meta_mem_sum is None else meta_mem_sum + meta_mem\n'
'                    pooled_sum = pooled.float().sum(0, keepdim=True) if pooled_sum is None else pooled_sum + pooled.float().sum(0, keepdim=True)\n'
'                    cond_sum = cond_vec.float().sum(0, keepdim=True) if cond_sum is None else cond_sum + cond_vec.float().sum(0, keepdim=True)\n'
'                    elem_sum = counts.float().sum(0, keepdim=True) if elem_sum is None else elem_sum + counts.float().sum(0, keepdim=True)\n'
'                    fp_sum = fp_p.float().sum(0) if fp_sum is None else fp_sum + fp_p.float().sum(0)\n'
'                    maccs_sum = maccs_p.float().sum(0) if maccs_sum is None else maccs_sum + maccs_p.float().sum(0)\n'
'                    n += len(items)\n'
'                    for it in items:\n'
"                        peak_mzs_list.append(torch.from_numpy(it['mz']).float().to(device))\n"
'                    # Gather valid fragment peak tokens without averaging misaligned m/z indices\n'
'                    valid_peaks = ~mask[:, 3:]\n'
'                    peak_tokens_list.append(memory[:, 3:][valid_peaks].float())\n'
"                    peak_intens_list.append(batch['peaks'][..., 64].to(device)[valid_peaks].float())\n"
'\n'
"                # Select top_k_peaks highest-intensity fragment embeddings across the molecule's spectra,\n"
'                # deduplicating peaks within 0.01 Da bins across replicate spectra\n'
'                all_peak_toks = torch.cat(peak_tokens_list, dim=0)\n'
'                all_peak_ints = torch.cat(peak_intens_list, dim=0)\n'
'                all_peak_mzs = torch.cat(peak_mzs_list, dim=0) if peak_mzs_list else torch.zeros(0, device=device)\n'
'                mol_peaks = torch.zeros((1, top_k_peaks, 320), dtype=torch.float32, device=device)\n'
'                mol_peak_mask = torch.ones((1, top_k_peaks), dtype=torch.bool, device=device)\n'
'                if len(all_peak_toks) > 0:\n'
'                    sorted_idx = torch.argsort(all_peak_ints, descending=True)\n'
'                    bins = (all_peak_mzs[sorted_idx] * 100).long()\n'
'                    _, unique_pos = np.unique(bins.cpu().numpy(), return_index=True)\n'
'                    dedup_idx = sorted_idx[torch.from_numpy(np.sort(unique_pos)).to(device)]\n'
'                    take = min(top_k_peaks, len(dedup_idx))\n'
'                    chosen = dedup_idx[:take]\n'
'                    mz_sort = torch.argsort(all_peak_mzs[chosen])\n'
'                    chosen = chosen[mz_sort]\n'
'                    mol_peaks[0, :take] = all_peak_toks[chosen]\n'
'                    mol_peak_mask[0, :take] = False\n'
'\n'
'                mol_mem = torch.cat([meta_mem_sum / n, mol_peaks], dim=1)\n'
'                mol_mask = torch.cat([torch.zeros((1, 3), dtype=torch.bool, device=device), mol_peak_mask], dim=1)\n'
'                memory, mask = model.conditioned_memory(\n'
'                    mol_mem, mol_mask, torch.tensor([mass], device=device), pooled_sum / n, cond_sum / n\n'
'                )\n'
'            memories.append(memory)\n'
'            masks.append(mask)\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [22]:
fragment = (
'            masses.append(mass)\n'
'            pred_fps.append((fp_sum / n).cpu().numpy())\n'
'            pred_maccs.append((maccs_sum / n).cpu().numpy())\n'
'            pred_elems.append((elem_sum.squeeze(0) / n).cpu().numpy())\n'
'\n'
"        with torch.amp.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):\n"
'            hypotheses = beam_search(model, torch.cat(memories), torch.cat(masks),\n'
'                                     beam_size=beam_size, max_len=max_len, top_k=top_k)\n'
'\n'
'        for (molecule_id, _), mass, p_fp, p_mc, p_el, paths in zip(\n'
'            chunk, masses, pred_fps, pred_maccs, pred_elems, hypotheses\n'
'        ):\n'
'            unique = {}\n'
'            p_fp_clip = np.clip(p_fp, 1e-6, 1.0 - 1e-6)\n'
'            fp_logits = np.log(p_fp_clip) - np.log1p(-p_fp_clip)\n'
'            fp_log_not_p = float(np.sum(-np.logaddexp(0.0, fp_logits)))\n'
'            p_fp_sum = float(p_fp.sum())\n'
'            p_fp_norm = float(np.linalg.norm(p_fp))\n'
'            p_mc_sum = float(p_mc.sum())\n'
'            p_mc_norm = float(np.linalg.norm(p_mc))\n'
'            for ids, log_prob in paths:\n'
'                smi = tokenizer.decode(ids)\n'
'                raw_mol = Chem.MolFromSmiles(smi)\n'
'                if raw_mol is None or raw_mol.GetNumAtoms() == 0:\n'
'                    continue\n'
'                mol = evalkit.standardize_mol(raw_mol) or raw_mol\n'
'                if any(a.GetSymbol() not in ELEMENTS_ORDER for a in mol.GetAtoms()):\n'
'                    continue\n'
'                key = evalkit.fast_key(mol)\n'
'                if not key:\n'
'                    continue\n'
'                ppm = abs(rdMolDescriptors.CalcExactMolWt(mol) - mass) / mass * 1e6\n'
'                # Full Self-Spec2FP Discriminative Reranking (Morgan4096 + MACCS167 + NB + Elem)\n'
'                c_fp = evalkit.compute_morgan_fp(mol).astype(np.float32)\n'
'                c_mc = evalkit.compute_maccs_fp(mol).astype(np.float32)\n'
'                c_fp_sum = float(c_fp.sum())\n'
'                c_mc_sum = float(c_mc.sum())\n'
'                dot_fp = float(c_fp @ p_fp)\n'
'                cos_sim = dot_fp / max(math.sqrt(c_fp_sum) * p_fp_norm, 1e-8)\n'
'                tan_sim = dot_fp / max(c_fp_sum + p_fp_sum - dot_fp, 1e-8)\n'
'                nb_ll = (float(c_fp @ fp_logits) + fp_log_not_p) / 4096.0\n'
'                dot_mc = float(c_mc @ p_mc)\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [23]:
fragment = (
'                maccs_cos = dot_mc / max(math.sqrt(c_mc_sum) * p_mc_norm, 1e-8)\n'
'                maccs_tan = dot_mc / max(c_mc_sum + p_mc_sum - dot_mc, 1e-8)\n'
'                elem_dist = float(np.abs(mol_elem_counts(mol) - p_el).sum())\n'
'                mass_score = -0.5 * (min(ppm, 50.0) / 7.5) ** 2\n'
'\n'
'                # Hybrid generative + Stage06-aligned discriminative score\n'
'                score = (\n'
'                    log_prob\n'
'                    + 2.0 * nb_ll\n'
'                    + 1.8 * cos_sim\n'
'                    + 1.8 * tan_sim\n'
'                    + 0.6 * maccs_cos\n'
'                    + 0.6 * maccs_tan\n'
'                    + 0.40 * mass_score\n'
'                    - 0.04 * elem_dist\n'
'                )\n'
'                candidate = dict(\n'
'                    molecule_id=str(molecule_id),\n'
'                    smiles=Chem.MolToSmiles(mol, isomericSmiles=False),\n'
'                    fast_key=key, score=score, log_prob=log_prob,\n'
'                    nb_ll=nb_ll, cos_sim=cos_sim, tan_sim=tan_sim,\n'
'                    maccs_cos=maccs_cos, maccs_tan=maccs_tan,\n'
"                    elem_dist=elem_dist, ppm_err=ppm, source='spec2smiles'\n"
'                )\n'
"                if key not in unique or score > unique[key]['score']:\n"
'                    unique[key] = candidate\n'
'\n'
'            # Tautomer-canonical deduplication on shortlist\n'
'            canon_seen = set()\n'
'            rank = 0\n'
"            for candidate in sorted(unique.values(), key=lambda x: -x['score']):\n"
"                ik14 = evalkit.canon_ik14(candidate['smiles'])\n"
'                if not ik14 or ik14 in canon_seen:\n'
'                    continue\n'
'                canon_seen.add(ik14)\n'
'                rank += 1\n'
'                candidate.update(rank=rank, ik14=ik14)\n'
'                rows.append(candidate)\n'
'                if rank >= top_k:\n'
'                    break\n'
'\n'
'    return pd.DataFrame(rows, columns=COLUMNS).astype({\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [24]:
fragment = (
"        'rank': 'int64', 'score': 'float64', 'log_prob': 'float64',\n"
"        'nb_ll': 'float64', 'cos_sim': 'float64', 'tan_sim': 'float64',\n"
"        'maccs_cos': 'float64', 'maccs_tan': 'float64',\n"
"        'elem_dist': 'float64', 'ppm_err': 'float64'\n"
'    })\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [25]:
module_path = WORK / 'evalkit.py'
module_path.write_text('', encoding='utf-8')

0

In [26]:
fragment = (
'"""\n'
'evalkit.py - Master Cheminformatics & Metric Toolkit for Enveda CASMI 2026.\n'
'Generated by Stage 02 (02_eval_splits.ipynb).\n'
'Single source of truth for all downstream stages (models, retrieval, reranker, submission).\n'
'"""\n'
'import os\n'
'import sys\n'
'import time\n'
'import math\n'
'import warnings\n'
'import functools\n'
'from typing import Dict, List, Tuple, Optional, Union, Any, Set\n'
'import numpy as np\n'
'import pandas as pd\n'
'\n'
'# RDKit imports\n'
'try:\n'
'    from rdkit import Chem, RDLogger\n'
'    from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem\n'
'    from rdkit.Chem.Scaffolds import MurckoScaffold\n'
'    from rdkit.Chem.MolStandardize import rdMolStandardize\n'
'    try:\n'
'        from rdkit.Chem import rdFingerprintGenerator\n'
'        _HAS_FP_GEN = True\n'
'    except ImportError:\n'
'        _HAS_FP_GEN = False\n'
'    RDLogger.DisableLog("rdApp.*")\n'
'    RDKIT_VERSION = Chem.rdBase.rdkitVersion\n'
'except ImportError as e:\n'
'    raise ImportError(f"RDKit is mandatory for CASMI 2026 evalkit: {e}")\n'
'\n'
'# Version verification\n'
'OFFICIAL_RDKIT_VERSION = "2026.03.3"\n'
'if RDKIT_VERSION != OFFICIAL_RDKIT_VERSION:\n'
'    warnings.warn(\n'
'        f"[evalkit] Official competition scorer pins RDKit {OFFICIAL_RDKIT_VERSION}; "\n'
'        f"current environment is {RDKIT_VERSION}. Tautomer canonicalization for rare "\n'
'        f"conjugated systems may produce minor edge-case discrepancies.",\n'
'        UserWarning\n'
'    )\n'
'\n'
'# ── 1. Shared Fingerprint Specification ─────────────────────────────────────\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [27]:
fragment = (
'FP_SPEC: Dict[str, Any] = {\n'
'    "morgan": {\n'
'        "radius": 2,\n'
'        "nBits": 4096,\n'
'        "chirality": False,\n'
'        "dtype": "uint8",\n'
'    },\n'
'    "maccs": {\n'
'        "nBits": 167,\n'
'        "dtype": "uint8",\n'
'    },\n'
'}\n'
'\n'
'# ── 2. Adduct Carrier Mass Invariant ─────────────────────────────────────────\n'
'CARRIER_MASSES: Dict[str, float] = {\n'
'    "[M+H]+":      1.007276,\n'
'    "[M-H]-":     -1.007276,\n'
'    "[M+Na]+":    22.989218,\n'
'    "[M+NH4]+":   18.033823,\n'
'    "[M+K]+":     38.963158,\n'
'    "[M+H-H2O]+": -17.002740,\n'
'    "[M+FA-H]-":  44.998201,\n'
'    "[M+Cl]-":    34.969402,\n'
'}\n'
'\n'
'def calc_neutral_mass(precursor_mz: float, adduct: str, polarity: str = "+") -> float:\n'
'    """\n'
'    Strictly compute neutral mass as: M_neutral = (precursor_mz * |z|) - carrier_mass\n'
'    Fallback: [M+H]+ for positive polarity, [M-H]- for negative polarity.\n'
'    """\n'
'    add_clean = str(adduct).strip() if adduct is not None else ""\n'
'    if add_clean in CARRIER_MASSES:\n'
'        carrier = CARRIER_MASSES[add_clean]\n'
'    else:\n'
'        pol_str = str(polarity).strip().lower()\n'
'        if pol_str in ["positive", "pos", "+", "1"]:\n'
'            carrier = CARRIER_MASSES["[M+H]+"]\n'
'        else:\n'
'            carrier = CARRIER_MASSES["[M-H]-"]\n'
'    return float(precursor_mz) - carrier\n'
'\n'
'# ── 3. Molecule Standardization & Desalting ─────────────────────────────────\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [28]:
fragment = (
'def standardize_mol(mol_or_smiles: Union[str, Chem.Mol]) -> Optional[Chem.Mol]:\n'
'    """\n'
'    Standardize a molecule for downstream evaluation and library indexing:\n'
'    - Retain largest organic fragment (heavy atoms >= 3).\n'
'    - Neutralize formal charges via rdMolStandardize.Uncharger.\n'
'    - Clear isotope labels.\n'
'    - Sanitize.\n'
'    - Filter heavy atoms in [3, 120].\n'
'    """\n'
'    if isinstance(mol_or_smiles, str):\n'
'        if not mol_or_smiles or mol_or_smiles.strip() == "":\n'
'            return None\n'
'        mol = Chem.MolFromSmiles(mol_or_smiles)\n'
'    else:\n'
'        mol = mol_or_smiles\n'
'    if mol is None:\n'
'        return None\n'
'    try:\n'
'        # Desalt: Largest fragment chooser\n'
'        lfc = rdMolStandardize.LargestFragmentChooser()\n'
'        mol = lfc.choose(mol)\n'
'        if mol is None:\n'
'            return None\n'
'        n_heavy = mol.GetNumHeavyAtoms()\n'
'        if n_heavy < 3 or n_heavy > 120:\n'
'            return None\n'
'        # Neutralize charges\n'
'        uncharger = rdMolStandardize.Uncharger()\n'
'        mol = uncharger.uncharge(mol)\n'
'        if mol is None:\n'
'            return None\n'
'        # Clear isotopes\n'
'        for atom in mol.GetAtoms():\n'
'            if atom.GetIsotope() != 0:\n'
'                atom.SetIsotope(0)\n'
'        # Sanitize\n'
'        Chem.SanitizeMol(mol)\n'
'        return mol\n'
'    except Exception:\n'
'        return None\n'
'\n'
'# ── 4. InChIKey Generation & Speed Guarded Canonicalization ───────────────────\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [29]:
fragment = (
'_IK14_CACHE: Dict[str, str] = {}\n'
'_FAST_IK14_CACHE: Dict[str, str] = {}\n'
'\n'
'def fast_key(mol_or_smiles: Union[str, Chem.Mol]) -> str:\n'
'    """\n'
'    Compute plain InChIKey14 without tautomer enumeration.\n'
'    Extremely fast for candidate filtering and initial deduplication.\n'
'    """\n'
'    if isinstance(mol_or_smiles, str):\n'
'        if mol_or_smiles in _FAST_IK14_CACHE:\n'
'            return _FAST_IK14_CACHE[mol_or_smiles]\n'
'        smi = mol_or_smiles\n'
'        mol = Chem.MolFromSmiles(smi)\n'
'    else:\n'
'        smi = None\n'
'        mol = mol_or_smiles\n'
'    if mol is None:\n'
'        return ""\n'
'    try:\n'
'        ik14 = Chem.MolToInchiKey(mol)[:14]\n'
'    except Exception:\n'
'        ik14 = ""\n'
'    if smi is not None:\n'
'        _FAST_IK14_CACHE[smi] = ik14\n'
'    return ik14\n'
'\n'
'# Tautomer enumerator singleton with speed guards\n'
'_TAUTOMER_ENUMERATOR = rdMolStandardize.TautomerEnumerator()\n'
'_TAUTOMER_ENUMERATOR.SetMaxTautomers(50)\n'
'_TAUTOMER_ENUMERATOR.SetMaxTransforms(100)\n'
'\n'
'def canon_ik14(mol_or_smiles: Union[str, Chem.Mol], timeout_sec: float = 1.5) -> str:\n'
'    """\n'
'    Compute official ground-truth canonical InChIKey14:\n'
'    - RDKit TautomerEnumerator canonicalization\n'
'    - Max tautomers = 50, max transforms = 100\n'
'    - Fallback to fast_key on timeout or error\n'
'    - In-memory cached\n'
'    """\n'
'    if isinstance(mol_or_smiles, str):\n'
'        smi = mol_or_smiles.strip()\n'
'        if smi in _IK14_CACHE:\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [30]:
fragment = (
'            return _IK14_CACHE[smi]\n'
'        mol = Chem.MolFromSmiles(smi)\n'
'    else:\n'
'        smi = None\n'
'        mol = mol_or_smiles\n'
'    if mol is None:\n'
'        return ""\n'
'    # Speed guarded execution\n'
'    t0 = time.time()\n'
'    try:\n'
'        canon_mol = _TAUTOMER_ENUMERATOR.Canonicalize(mol)\n'
'        if time.time() - t0 > timeout_sec:\n'
'            ik14 = Chem.MolToInchiKey(mol)[:14]\n'
'        else:\n'
'            ik14 = Chem.MolToInchiKey(canon_mol)[:14]\n'
'    except Exception:\n'
'        ik14 = Chem.MolToInchiKey(mol)[:14]\n'
'    if smi is not None:\n'
'        _IK14_CACHE[smi] = ik14\n'
'    return ik14\n'
'\n'
'# ── 5. Fingerprint Generators ────────────────────────────────────────────────\n'
'if _HAS_FP_GEN:\n'
'    _MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)\n'
'else:\n'
'    _MORGAN_GEN = None\n'
'\n'
'def compute_morgan_fp(mol_or_smiles: Union[str, Chem.Mol], n_bits: int = 4096, radius: int = 2) -> np.ndarray:\n'
'    """Compute binary Morgan fingerprint as uint8 numpy vector of shape (n_bits,)."""\n'
'    if isinstance(mol_or_smiles, str):\n'
'        mol = Chem.MolFromSmiles(mol_or_smiles)\n'
'    else:\n'
'        mol = mol_or_smiles\n'
'    if mol is None:\n'
'        return np.zeros(n_bits, dtype=np.uint8)\n'
'    if _MORGAN_GEN is not None and n_bits == 4096 and radius == 2:\n'
'        return _MORGAN_GEN.GetFingerprintAsNumPy(mol).astype(np.uint8)\n'
'    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits, useChirality=False)\n'
'    arr = np.zeros(n_bits, dtype=np.uint8)\n'
'    for bit in fp.GetOnBits():\n'
'        arr[bit] = 1\n'
'    return arr\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [31]:
fragment = (
'\n'
'def compute_maccs_fp(mol_or_smiles: Union[str, Chem.Mol]) -> np.ndarray:\n'
'    """Compute binary MACCS keys fingerprint as uint8 numpy vector of shape (167,)."""\n'
'    if isinstance(mol_or_smiles, str):\n'
'        mol = Chem.MolFromSmiles(mol_or_smiles)\n'
'    else:\n'
'        mol = mol_or_smiles\n'
'    if mol is None:\n'
'        return np.zeros(167, dtype=np.uint8)\n'
'    fp = rdMolDescriptors.GetMACCSKeysFingerprint(mol)\n'
'    arr = np.zeros(167, dtype=np.uint8)\n'
'    for bit in fp.GetOnBits():\n'
'        arr[bit] = 1\n'
'    return arr\n'
'\n'
'# ── 6. Bemis-Murcko Scaffold ─────────────────────────────────────────────────\n'
'def get_murcko_scaffold(mol_or_smiles: Union[str, Chem.Mol]) -> str:\n'
'    """Compute Bemis-Murcko scaffold SMILES without chirality, with acyclic formula fallback."""\n'
'    if isinstance(mol_or_smiles, str):\n'
'        mol = Chem.MolFromSmiles(mol_or_smiles)\n'
'    else:\n'
'        mol = mol_or_smiles\n'
'    if mol is None:\n'
'        return ""\n'
'    try:\n'
'        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)\n'
'        if scaf and len(scaf) > 0:\n'
'            return scaf\n'
'        form = rdMolDescriptors.CalcMolFormula(mol)\n'
'        return f"_acyclic_{form}"\n'
'    except Exception:\n'
'        return "_unknown"\n'
'\n'
'# ── 7. Competition Metric: MRR@25 ────────────────────────────────────────────\n'
'def mrr_at_25(pred: Dict[str, List[str]], truth: Dict[str, str]) -> float:\n'
'    """\n'
'    Exact local replica of official CASMI 2026 MRR@25 competition metric.\n'
'    - Evaluates raw order of predictions, top 25 guesses per molecule_id.\n'
"    - Reciprocal rank = 1.0 / pos of first guess whose canon_ik14 equals truth's canon_ik14.\n"
'    - Returns mean reciprocal rank across all ground truth query IDs.\n'
'    """\n'
'    scores: List[float] = []\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [32]:
fragment = (
'    for qid, t_val in truth.items():\n'
'        t_key = t_val if (len(t_val) == 14 and t_val.isupper()) else canon_ik14(t_val)\n'
'        guesses = pred.get(qid, [])[:25]\n'
'        rr = 0.0\n'
'        for i, g in enumerate(guesses, start=1):\n'
'            g_key = g if (len(g) == 14 and g.isupper()) else canon_ik14(g)\n'
'            if g_key == t_key:\n'
'                rr = 1.0 / i\n'
'                break\n'
'        scores.append(rr)\n'
'    return float(np.mean(scores)) if len(scores) > 0 else 0.0\n'
'\n'
'def mrr_at_25_dedup(pred: Dict[str, List[str]], truth: Dict[str, str]) -> Tuple[float, float, int]:\n'
'    """\n'
'    Diagnostic variant evaluating MRR@25 after InChIKey14 deduplication.\n'
'    Returns:\n'
'        (mrr_raw, mrr_dedup, total_wasted_ranks)\n'
'    """\n'
'    raw_scores: List[float] = []\n'
'    dedup_scores: List[float] = []\n'
'    total_wasted = 0\n'
'    for qid, t_val in truth.items():\n'
'        t_key = t_val if (len(t_val) == 14 and t_val.isupper()) else canon_ik14(t_val)\n'
'        raw_guesses = pred.get(qid, [])[:25]\n'
'        # Raw MRR\n'
'        rr_raw = 0.0\n'
'        for i, g in enumerate(raw_guesses, start=1):\n'
'            g_key = g if (len(g) == 14 and g.isupper()) else canon_ik14(g)\n'
'            if g_key == t_key:\n'
'                rr_raw = 1.0 / i\n'
'                break\n'
'        raw_scores.append(rr_raw)\n'
'        # Deduplicate guesses by InChIKey14 while preserving order\n'
'        seen_keys: Set[str] = set()\n'
'        dedup_guesses: List[str] = []\n'
'        for g in raw_guesses:\n'
'            g_key = g if (len(g) == 14 and g.isupper()) else canon_ik14(g)\n'
'            if g_key not in seen_keys:\n'
'                seen_keys.add(g_key)\n'
'                dedup_guesses.append(g)\n'
'        total_wasted += (len(raw_guesses) - len(dedup_guesses))\n'
'        # Dedup MRR\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [33]:
fragment = (
'        rr_dedup = 0.0\n'
'        for i, g in enumerate(dedup_guesses[:25], start=1):\n'
'            g_key = g if (len(g) == 14 and g.isupper()) else canon_ik14(g)\n'
'            if g_key == t_key:\n'
'                rr_dedup = 1.0 / i\n'
'                break\n'
'        dedup_scores.append(rr_dedup)\n'
'    mrr_raw = float(np.mean(raw_scores)) if raw_scores else 0.0\n'
'    mrr_dedup = float(np.mean(dedup_scores)) if dedup_scores else 0.0\n'
'    return mrr_raw, mrr_dedup, total_wasted\n'
'\n'
'# ── 8. Submission Validator ──────────────────────────────────────────────────\n'
'def validate_submission(\n'
'    df_or_path: Union[pd.DataFrame, str],\n'
'    test_ids: Union[List[str], Set[str], pd.Series]\n'
') -> Tuple[bool, List[str], List[str], Dict[str, Any]]:\n'
'    """\n'
'    Rigorous validator implementing all competition rejection rules:\n'
"    - Column contract: ['molecule_id', 'smiles']\n"
'    - 100% test ID completeness (zero missing, zero duplicates)\n'
'    - Non-null, non-empty SMILES string\n'
"    - Up to 25 guesses per query joined by ';'\n"
'    - RDKit parseability of every individual SMILES guess\n'
'    - Detection of internal duplicate InChIKey14s (warning)\n'
'    Returns:\n'
'        (is_valid, errors, warnings, summary_stats)\n'
'    """\n'
'    errors: List[str] = []\n'
'    warnings_list: List[str] = []\n'
'    summary: Dict[str, Any] = {}\n'
'\n'
'    if isinstance(df_or_path, str):\n'
'        if not os.path.exists(df_or_path):\n'
'            return False, [f"Submission file does not exist: {df_or_path}"], [], {}\n'
'        df = pd.read_csv(df_or_path)\n'
'    else:\n'
'        df = df_or_path.copy()\n'
'\n'
'    # Column contract\n'
'    expected_cols = {"molecule_id", "smiles"}\n'
'    if not expected_cols.issubset(set(df.columns)):\n'
'        missing = expected_cols - set(df.columns)\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [34]:
fragment = (
'        errors.append(f"Missing required columns: {missing}. Found: {list(df.columns)}")\n'
'        return False, errors, warnings_list, summary\n'
'\n'
'    # Row completeness\n'
'    test_set = set(str(x) for x in test_ids)\n'
'    sub_set = set(df["molecule_id"].astype(str))\n'
'    if len(df) != len(test_set):\n'
'        errors.append(f"Row count mismatch: submission has {len(df)} rows, expected {len(test_set)}")\n'
'    missing_ids = test_set - sub_set\n'
'    if len(missing_ids) > 0:\n'
'        errors.append(f"Missing {len(missing_ids)} test molecule_ids! Example missing: {list(missing_ids)[:5]}")\n'
'    extra_ids = sub_set - test_set\n'
'    if len(extra_ids) > 0:\n'
'        errors.append(f"Found {len(extra_ids)} unexpected molecule_ids not in test set!")\n'
'    if df["molecule_id"].duplicated().any():\n'
'        n_dups = int(df["molecule_id"].duplicated().sum())\n'
'        errors.append(f"Duplicate molecule_id entries found in submission: {n_dups} duplicates")\n'
'\n'
'    # Content checks\n'
'    null_smiles = int(df["smiles"].isna().sum())\n'
'    if null_smiles > 0:\n'
'        errors.append(f"Found {null_smiles} null/NaN values in \'smiles\' column")\n'
'\n'
'    total_guesses = 0\n'
'    total_unparseable = 0\n'
'    total_internal_dups = 0\n'
'    max_guesses_seen = 0\n'
'\n'
'    for idx, row in df.iterrows():\n'
'        s_val = str(row["smiles"]) if pd.notna(row["smiles"]) else ""\n'
'        if len(s_val.strip()) == 0:\n'
'            errors.append(f"Empty SMILES string for molecule_id \'{row[\'molecule_id\']}\' at row {idx}")\n'
'            continue\n'
'        parts = [p.strip() for p in s_val.split(";") if p.strip()]\n'
'        n_p = len(parts)\n'
'        max_guesses_seen = max(max_guesses_seen, n_p)\n'
'        total_guesses += n_p\n'
'        if n_p > 25:\n'
'            errors.append(f"Row {idx} (\'{row[\'molecule_id\']}\') has {n_p} guesses; maximum permitted is 25")\n'
'        # Check parseability and duplicates\n'
'        seen_keys: Set[str] = set()\n'
'        for p in parts:\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [35]:
fragment = (
'            mol = Chem.MolFromSmiles(p)\n'
'            if mol is None:\n'
'                total_unparseable += 1\n'
'                if total_unparseable <= 5:\n'
'                    errors.append(f"Invalid unparseable SMILES \'{p}\' in row {idx} (\'{row[\'molecule_id\']}\')")\n'
'            else:\n'
'                k = fast_key(mol)\n'
'                if k in seen_keys:\n'
'                    total_internal_dups += 1\n'
'                seen_keys.add(k)\n'
'\n'
'    if total_unparseable > 5:\n'
'        errors.append(f"Total unparseable SMILES across submission: {total_unparseable}")\n'
'    if total_internal_dups > 0:\n'
'        warnings_list.append(\n'
'            f"Found {total_internal_dups} duplicate InChIKey14 guesses inside prediction lists! "\n'
'            f"These waste ranking positions without improving MRR@25. Strictly deduplicate before saving."\n'
'        )\n'
'\n'
'    is_valid = (len(errors) == 0)\n'
'    summary = {\n'
'        "is_valid": is_valid,\n'
'        "n_rows": len(df),\n'
'        "expected_rows": len(test_set),\n'
'        "total_guesses": total_guesses,\n'
'        "avg_guesses_per_mol": total_guesses / max(len(df), 1),\n'
'        "max_guesses_seen": max_guesses_seen,\n'
'        "total_unparseable": total_unparseable,\n'
'        "total_internal_dups": total_internal_dups,\n'
'        "n_errors": len(errors),\n'
'        "n_warnings": len(warnings_list),\n'
'    }\n'
'    return is_valid, errors, warnings_list, summary\n'
)
with module_path.open('a', encoding='utf-8') as f:
    f.write(fragment)

In [36]:
encoder_path = find_file('spec2fp_model.pt')
encoder_config_path = find_file('spec2fp_config.json', encoder_path.parent)
upstream = json.loads(encoder_config_path.read_text())
if upstream.get('training', {}).get('FULL_TRAIN', False):
    raise ValueError('Full-train encoder contaminates validation; attach train-only weights')
import importlib
for name in ('evalkit', 'spec2fp', 'spec2smiles'):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
import evalkit, spec2fp, spec2smiles as s2s
train_path = find_file('train_clean.parquet')
test_path = find_file('test_clean.parquet')
split_path = find_file('splits.parquet')
splits = pd.read_parquet(split_path, columns=['molecule_id', 'split3'])
splits['molecule_id'] = splits.molecule_id.astype(str)
assert not splits.molecule_id.duplicated().any()
assert set(splits.split3) == {'train', 'val_model', 'val_rank'}
split_map = splits.set_index('molecule_id').split3
cfg = dict(encoder=upstream['model'], max_length=128, decoder_layers=4,
           d_model=320, nhead=8, dim_feedforward=1280, dropout=0.1,
           norm_first=True, top_k_peaks=128, mz_dim=64, epochs=10,
           time_budget_s=10.0 * 3600,
           lr_enc=8e-5, lr_dec=4e-4, batch_size=256, warmup_fraction=0.05, ema_decay=0.999,
           label_smoothing=0.05, seed=42, element_conditioning='predicted',
           elements_order=spec2fp.ELEMENTS_ORDER)
cfg['top_k_peaks'] = upstream.get('training', {}).get('TOP_K_PEAKS', 128)
cfg['mz_dim'] = (cfg['encoder'].get('feat_dim', 129)-1)//2
for key, actual in [('adduct_vocab', spec2fp.ADDUCT_VOCAB),
                    ('inst_vocab', spec2fp.INST_VOCAB),
                    ('elements_order', spec2fp.ELEMENTS_ORDER)]:
    assert upstream.get(key, actual) == actual, f'Checkpoint vocabulary mismatch: {key}'

/kaggle/working/evalkit.py:35: UserWarning: [evalkit] Official competition scorer pins RDKit 2026.03.3; current environment is 2026.09.1. Tautomer canonicalization for rare conjugated systems may produce minor edge-case discrepancies.
  warnings.warn(


Vocabulary uses training molecules only. Invalid or >126-token targets are
excluded rather than truncated; validation generation still covers every molecule.
Randomized SMILES that exceed the budget or vocabulary fall back to canonical targets.
Vocabulary size is data dependent (bracketed atoms are individual tokens).

In [37]:
meta = pd.read_parquet(train_path, columns=['molecule_id', 'smiles'], use_threads=False)
meta['molecule_id'] = meta.molecule_id.astype(str)
smi_counts = meta.dropna(subset=['smiles']).groupby(['molecule_id', 'smiles'], sort=False).size().reset_index(name='cnt')
meta = smi_counts.sort_values(['molecule_id', 'cnt'], ascending=[True, False]).drop_duplicates('molecule_id')[['molecule_id', 'smiles']].copy()
del smi_counts
meta['split3'] = meta.molecule_id.map(split_map)
assert meta.split3.notna().all(), 'Missing split assignments'
canonical = {}
probe = s2s.SmilesTokenizer()
for smi in meta.smiles.dropna().unique():
    mol = Chem.MolFromSmiles(smi)
    if mol is None or any(a.GetSymbol() not in spec2fp.ELEMENTS_ORDER for a in mol.GetAtoms()):
        continue
    value = Chem.MolToSmiles(mol, isomericSmiles=False)
    try:
        probe.encode(value)
        canonical[smi] = value
    except ValueError:
        pass
train_smiles = {canonical[s] for s in meta.loc[meta.split3.eq('train'), 'smiles'] if s in canonical}
tokenizer = s2s.SmilesTokenizer().fit(sorted(train_smiles))
tokenizer.save(OUT / 'smiles_vocab.json')
smiles = sorted(set(canonical.values()))
smi_index = {s: i for i, s in enumerate(smiles)}
mol_to_canon = {r.molecule_id: canonical[r.smiles] for r in meta.itertuples() if r.smiles in canonical}
target_map = {mid: smi_index[c_smi] for mid, c_smi in mol_to_canon.items()}
dtype = np.uint8 if len(tokenizer.vocab) <= 256 else np.int16
assert len(tokenizer.vocab) < 32768
tokens = np.asarray([tokenizer.encode(s) for s in smiles], dtype=dtype)
num_aug = 4
tokens_aug = np.repeat(tokens[None, :, :], num_aug, axis=0)
for i, smi in enumerate(smiles):
    mol = Chem.MolFromSmiles(smi)
    for k in range(num_aug):
        randomized = Chem.MolToSmiles(mol, doRandom=True, isomericSmiles=False)
        try:
            encoded = tokenizer.encode(randomized)
            if 3 not in encoded:
                tokens_aug[k, i] = encoded
        except ValueError:
            pass
cfg['vocab_size'] = len(tokenizer.vocab)
cfg['target_shape'] = list(tokens.shape)
cfg['target_bytes'] = tokens.nbytes + tokens_aug.nbytes
cfg['excluded_unique_smiles'] = int(meta.smiles.nunique()-len(canonical))
del canonical, smi_index, train_smiles, probe, mol_to_canon
gc.collect()
print('Vocabulary / target MB / RSS GB:', cfg['vocab_size'], tokens.nbytes/1e6, check_ram())

Vocabulary / target MB / RSS GB: 73 35.113216 1.363279872


In [38]:
paths = {name: CACHE / f'{name}.bin' for name in ('train', 'val_model')}
streams = {name: path.open('wb') for name, path in paths.items()}
try:
    for record_batch in pq.ParquetFile(train_path).iter_batches(batch_size=2048):
        train_df = record_batch.to_pandas()
        train_df['molecule_id'] = train_df.molecule_id.astype(str)
        train_df['split3'] = train_df.molecule_id.map(split_map)
        train_df['ce'] = pd.to_numeric(train_df.ce, errors='coerce').fillna(0)
        train_split = train_df.loc[train_df.split3.eq('train') & train_df.molecule_id.isin(target_map)]
        val_split = train_df.loc[train_df.split3.eq('val_model') & train_df.molecule_id.isin(target_map)]
        del train_df
        s2s.append_records(train_split, streams['train'], cfg, target_map)
        s2s.append_records(val_split, streams['val_model'], cfg, target_map)
        del train_split, val_split, record_batch
        check_ram()
finally:
    for stream in streams.values():
        stream.close()
ds_train = s2s.SpectrumDataset(paths['train'], cfg, tokenizer, smiles, tokens, augment=True, tokens_aug=tokens_aug)
ds_val = s2s.SpectrumDataset(paths['val_model'], cfg, tokenizer, smiles, tokens)
workers = 2 if torch.cuda.is_available() else 0
train_loader = torch.utils.data.DataLoader(ds_train, batch_size=cfg['batch_size'],
    shuffle=True, num_workers=workers, pin_memory=bool(workers), persistent_workers=bool(workers), collate_fn=ds_train.collate)
val_loader = torch.utils.data.DataLoader(ds_val, batch_size=cfg['batch_size'],
    shuffle=False, num_workers=workers, pin_memory=bool(workers), persistent_workers=bool(workers), collate_fn=ds_val.collate)
del target_map
gc.collect()
print('Training / validation spectra:', len(ds_train), len(ds_val))

Training / validation spectra: 2030434 240769


In [39]:
cfg['epochs'] = 10
TOTAL_BUDGET_S = 10.0 * 3600
INFER_RESERVE_S = 1.2 * 3600
TRAIN_BUDGET_S = TOTAL_BUDGET_S - INFER_RESERVE_S
model = s2s.Spec2SmilesModel(cfg)
model.warm_start(encoder_path)
model.to(device)
enc_params = model.encoder_parameters()
enc_ids = {id(p) for p in enc_params}
dec_params = [p for p in model.parameters() if id(p) not in enc_ids and p.requires_grad]
optimizer = torch.optim.AdamW([{'params': enc_params, 'lr': cfg['lr_enc']},
                              {'params': dec_params, 'lr': cfg['lr_dec']}], weight_decay=0.01)
total_steps = cfg['epochs'] * len(train_loader)
warmup_steps = max(1, int(total_steps * cfg['warmup_fraction']))
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
    return max(0.05, 0.5 * (1 + np.cos(np.pi * progress)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
ema = s2s.EMA(model, decay=cfg['ema_decay'])
parallel = nn.DataParallel(model) if torch.cuda.device_count() >= 2 else model
history, best = [], float('inf')
for epoch in range(cfg['epochs']):
    ep_start = time.time()
    for param in enc_params:
        param.requires_grad_(epoch >= 1)
    train_loss = s2s.run_epoch(parallel, train_loader, device, optimizer, scaler, ema, scheduler)
    val_loss = s2s.run_epoch(ema.model, val_loader, device)
    ep_time = time.time() - ep_start
    elapsed = time.time() - START
    history.append(dict(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss,
                        ep_seconds=round(ep_time, 1), elapsed_seconds=round(elapsed, 1),
                        lrs=[g['lr'] for g in optimizer.param_groups], rss_gb=check_ram()))
    if val_loss < best:
        best = val_loss
        torch.save({'model_state_dict': ema.model.state_dict(), 'epoch': epoch+1, 'val_loss': best},
                   OUT / 'spec2smiles_model.pt')
    print(history[-1], flush=True)
    if elapsed + ep_time * 1.15 > TRAIN_BUDGET_S and (epoch + 1) >= 4:
        print(f'Time guard reached at epoch {epoch+1} ({elapsed/3600:.2f}h elapsed); proceeding to beam search.')
        break
(OUT / 'spec2smiles_config.json').write_text(json.dumps(cfg, indent=2))
del parallel, optimizer, scheduler, scaler, ema, train_loader, val_loader
del ds_train, ds_val, tokens, tokens_aug, smiles, enc_params, dec_params
gc.collect()
torch.cuda.empty_cache()
checkpoint = torch.load(OUT / 'spec2smiles_model.pt', map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
best_epoch = checkpoint['epoch']
del checkpoint
model.eval()

/kaggle/working/spec2fp.py:143: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)


{'epoch': 1, 'train_loss': 1.1731897728916998, 'val_loss': 0.8553157786496653, 'ep_seconds': 2048.0, 'elapsed_seconds': 5055.2, 'lrs': [np.float64(7.94544521361089e-05), np.float64(0.0003972722606805445)], 'rss_gb': 2.925535232}


{'epoch': 2, 'train_loss': 0.9019520262616688, 'val_loss': 0.7922760505585971, 'ep_seconds': 3104.3, 'elapsed_seconds': 8159.6, 'lrs': [np.float64(7.518053734835386e-05), np.float64(0.0003759026867417693)], 'rss_gb': 2.975842304}


{'epoch': 3, 'train_loss': 0.8568953473854969, 'val_loss': 0.7721547740461955, 'ep_seconds': 3104.2, 'elapsed_seconds': 11263.9, 'lrs': [np.float64(6.709862357427337e-05), np.float64(0.0003354931178713668)], 'rss_gb': 2.976903168}


{'epoch': 4, 'train_loss': 0.8311212474104887, 'val_loss': 0.761921381444304, 'ep_seconds': 3102.3, 'elapsed_seconds': 14366.3, 'lrs': [np.float64(5.608156051632437e-05), np.float64(0.0002804078025816218)], 'rss_gb': 2.977984512}


{'epoch': 5, 'train_loss': 0.8120904212860995, 'val_loss': 0.7562547222438152, 'ep_seconds': 3102.0, 'elapsed_seconds': 17468.3, 'lrs': [np.float64(4.3324778668984354e-05), np.float64(0.00021662389334492174)], 'rss_gb': 2.978996224}


... (2 outputs left out)

{'epoch': 8, 'train_loss': 0.7760290679289736, 'val_loss': 0.7513013894795939, 'ep_seconds': 3100.9, 'elapsed_seconds': 26770.4, 'lrs': [np.float64(8.456927248479632e-06), np.float64(4.228463624239816e-05)], 'rss_gb': 2.98215424}


{'epoch': 9, 'train_loss': 0.7702796699090197, 'val_loss': 0.7515374402960411, 'ep_seconds': 3098.9, 'elapsed_seconds': 29869.4, 'lrs': [4.000000000000001e-06, 2e-05], 'rss_gb': 2.981126144}


Time guard reached at epoch 9 (8.30h elapsed); proceeding to beam search.


Spec2SmilesModel(
  (peak_proj): Sequential(
    (0): Linear(in_features=129, out_features=320, bias=True)
    (1): GELU(approximate='none')
    (2): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
    (3): Dropout(p=0.1, inplace=False)
  )
  (adduct_emb): Embedding(18, 320, padding_idx=0)
  (inst_emb): Embedding(9, 320, padding_idx=0)
  (pol_emb): Embedding(4, 320, padding_idx=0)
  (ce_proj): Sequential(
    (0): Linear(in_features=1, out_features=320, bias=True)
    (1): GELU(approximate='none')
  )
  (prec_proj): Sequential(
    (0): Linear(in_features=1, out_features=320, bias=True)
    (1): GELU(approximate='none')
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-4): 5 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=320, out_features=320, bias=True)
        )
        (linear1): Linear(in_features=320, out_features=1280, bias=True)
        (dropout): Dropout(p=0.1, inp

Cross-attend uncorrupted fragment peak embeddings across each molecule's spectra,
then decode 64 molecules together with beam width 16 and a 110-token budget. Completed
paths are scored with Self-Spec2FP discriminative reranking (Morgan & MACCS Tanimoto,
elemental L1 distance, ppm mass error) followed by tautomer-canonical deduplication.
Partition each source Parquet in one streaming pass; all spectra for an ID share one partition.

In [40]:
required = ['molecule_id', 'mz', 'inten', 'precursor_mz', 'neutral_mass',
            'adduct', 'ce', 'instrument', 'polarity']
partitions = {name: [] for name in ('val_model', 'val_rank', 'test')}
for source_path, is_test in ((train_path, False), (test_path, True)):
    writers = {}
    try:
        for rb in pq.ParquetFile(source_path).iter_batches(batch_size=8192, columns=required):
            frame = rb.to_pandas()
            frame['molecule_id'] = frame.molecule_id.astype(str)
            frame['ce'] = pd.to_numeric(frame.ce, errors='coerce').fillna(0)
            labels = pd.Series('test', index=frame.index) if is_test else frame.molecule_id.map(split_map)
            buckets = pd.util.hash_pandas_object(frame.molecule_id, index=False) % 64
            for (name, bucket), group in frame.groupby([labels, buckets], sort=False):
                if name not in partitions:
                    continue
                key = (name, int(bucket))
                table = pa.Table.from_pandas(group, preserve_index=False)
                if key not in writers:
                    part = CACHE / f'query_{name}_{bucket}.parquet'
                    writers[key] = pq.ParquetWriter(part, table.schema, compression='zstd')
                    partitions[name].append(part)
                writers[key].write_table(table)
            check_ram()
    finally:
        for writer in writers.values():
            writer.close()
del frame, group, table, rb
gc.collect()

0

In [ ]:
float_cols = {'score', 'log_prob', 'nb_ll', 'cos_sim', 'tan_sim', 'maccs_cos', 'maccs_tan', 'elem_dist', 'ppm_err'}
schema = pa.schema([(c, pa.int64() if c == 'rank' else pa.float64()
                     if c in float_cols else pa.string()) for c in s2s.COLUMNS])
summaries = {}
required = ['molecule_id', 'mz', 'inten', 'precursor_mz', 'neutral_mass',
            'adduct', 'ce', 'instrument', 'polarity']
for name in ('val_model', 'val_rank', 'test'):
    truth = {}
    if name != 'test':
        truth_frame = meta.loc[meta.split3.eq(name)].drop_duplicates('molecule_id')
        truth = {r.molecule_id: evalkit.canon_ik14(r.smiles) for r in truth_frame.itertuples()}
    rows, covered, reciprocal, hits, queries = 0, 0, 0.0, 0, 0
    with pq.ParquetWriter(OUT / f'cand_gen_{name}.parquet', schema, compression='zstd') as writer:
        for part in partitions[name]:
            query = pd.read_parquet(part, columns=required, use_threads=False)
            queries += query.molecule_id.nunique()
            query['molecule_id'] = query.molecule_id.astype(str)
            query['ce'] = pd.to_numeric(query.ce, errors='coerce').fillna(0)
            candidates = s2s.generate_candidates(model, tokenizer, query, device, beam_size=16, top_k=25, mol_batch_size=64, max_len=124)
            writer.write_table(pa.Table.from_pandas(candidates, schema=schema, preserve_index=False))
            rows += len(candidates)
            covered += candidates.molecule_id.nunique()
            if truth:
                for qid, frame in candidates.groupby('molecule_id'):
                    t_ik = truth.get(str(qid))
                    if t_ik:
                        matched = frame.loc[frame.ik14.eq(t_ik), 'rank']
                        if len(matched):
                            reciprocal += 1/int(matched.min())
                            hits += 1
            del query, candidates
            gc.collect()
            check_ram()
    summaries[name] = dict(queries=queries, candidates=rows, covered=covered,
                           empty_queries=queries-covered)
    if name != 'test':
        summaries[name].update(mrr25=reciprocal/max(queries, 1), recall25=hits/max(queries, 1))
    print(name, summaries[name])
test_cand = pd.read_parquet(OUT / 'cand_gen_test.parquet')
test_mol_ids = pd.read_parquet(test_path, columns=['molecule_id']).molecule_id.astype(str).unique().tolist()
test_preds = {str(qid): ';'.join(grp.sort_values('rank').head(25)['smiles']) for qid, grp in test_cand.groupby('molecule_id', sort=False)}
fallback_smi = 'CC(=O)Oc1ccccc1C(=O)O'
sub_df = pd.DataFrame({'molecule_id': test_mol_ids, 'smiles': [test_preds.get(mid) or fallback_smi for mid in test_mol_ids]})
is_valid, sub_errors, sub_warnings, sub_stats = evalkit.validate_submission(sub_df, test_mol_ids)
assert is_valid, f'Invalid submission_spec2smiles.csv: {sub_errors}'
sub_df.to_csv(OUT / 'submission_spec2smiles.csv', index=False)
print('Standalone submission validation:', sub_stats)

In [ ]:
metrics = dict(best_epoch=best_epoch, best_val_loss=best, history=history,
               splits=summaries, selection='val_model smoothed cross entropy',
               generation='beam16, max_len124, mol_batch64, top25, self-spec2fp reranked')
(OUT / 'spec2smiles_val_metrics.json').write_text(json.dumps(metrics, indent=2))
for name in ('spec2smiles.py', 'spec2fp.py', 'evalkit.py'):
    shutil.copy2(WORK / name, OUT / name)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

vm_ids = splits.loc[splits.split3.eq('val_model'), 'molecule_id'].tolist()
vr_ids = splits.loc[splits.split3.eq('val_rank'), 'molecule_id'].tolist()
manifest = dict(heldout_ids_hash=hashlib.sha256(''.join(sorted(vm_ids + vr_ids)).encode()).hexdigest(),
                stage='spec2smiles', trained_on_split='train', selected_on='val_model',
                untouched_evaluation='val_rank', seed=42, elapsed_seconds=time.time()-START,
                encoder=str(encoder_path), encoder_sha256=sha256(encoder_path),
                encoder_config_sha256=sha256(encoder_config_path),
                split_sha256=sha256(split_path), torch_version=torch.__version__,
                rdkit_version=Chem.rdBase.rdkitVersion, summaries=summaries,
                submission_validation=sub_stats,
                files={p.name: dict(bytes=p.stat().st_size, sha256=sha256(p))
                       for p in OUT.iterdir() if p.is_file() and p.name != 'manifest.json'})
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('Stage 07 complete:', OUT)